# Guide intelligent du Musée · 02 Baseline (jalon `m1`)

Deux modèles de référence pour reconnaître le style d'une œuvre : une régression logistique, le modèle classique à battre, puis un réseau dense écrit avec les briques de PyTorch.

Contexte du projet, données et découpage : voir `01_donnees.ipynb`.

## 1. Mise en place

In [ ]:
import platform
import time

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression

from commun import (MODELS_DIR, get_device, load_images, load_metadata, normalization_stats, set_seed,
                    split_indices, standardize)

set_seed()
DEVICE = get_device()
meta, CLASSES = load_metadata()
X_all, y_all = load_images(meta)
splits = split_indices(meta)
MEAN, STD = normalization_stats(X_all, splits["train"])

print(f"PyTorch {torch.__version__}, calculs sur : {DEVICE}")
print(f"{len(meta)} œuvres, {len(CLASSES)} styles ; " + ", ".join(f"{k} : {len(i)}" for k, i in splits.items()))

## 2. Régression logistique de référence

Le modèle classique à battre : une régression logistique multinomiale sur les pixels aplatis (3 × 128 × 128 = 49 152 valeurs), standardisés avec les statistiques du train. Avec 49 152 variables pour 900 œuvres d'entraînement, le modèle peut apprendre le train par cœur : la force de la régularisation, `C` (plus il est petit, plus elle est forte), est choisie sur la validation. Le test n'est pas utilisé.

In [ ]:
def flat_pixels(idx: np.ndarray) -> np.ndarray:
    """Œuvres d'un split, standardisées puis aplaties en vecteurs de 49 152 valeurs."""
    return standardize(X_all[idx], MEAN, STD).flatten(1).numpy()


X_train, X_val = flat_pixels(splits["train"]), flat_pixels(splits["val"])
y_train, y_val = y_all[splits["train"]].numpy(), y_all[splits["val"]].numpy()
print(f"X_train : {X_train.shape}, X_val : {X_val.shape}")

### 2.1 Choix de la régularisation

Une valeur de `C` par ordre de grandeur, resserrée autour de l'optimum.

In [ ]:
C_GRID = [1e-7, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 1e-3, 1e-2]
MAX_ITER = 2000

rows = []
for C in C_GRID:
    t0 = time.perf_counter()
    clf = LogisticRegression(C=C, max_iter=MAX_ITER).fit(X_train, y_train)
    assert clf.n_iter_[0] < MAX_ITER, f"C={C} : pas de convergence"
    rows.append({"C": C, "acc_train": clf.score(X_train, y_train), "acc_val": clf.score(X_val, y_val),
                 "entrainement_s": time.perf_counter() - t0})

grid = pd.DataFrame(rows).set_index("C")
grid.rename(index="{:g}".format).round(3)

### 2.2 Modèle retenu

En cas d'égalité sur la validation, la régularisation la plus forte l'emporte. Le temps d'inférence est mesuré image par image, comme sur une borne, prétraitement compris.

In [ ]:
best_C = grid["acc_val"].idxmax()  # premier maximum = plus petit C, la grille étant croissante

t0 = time.perf_counter()
logreg = LogisticRegression(C=best_C, max_iter=MAX_ITER).fit(X_train, y_train)
fit_s = time.perf_counter() - t0

acc_val = logreg.score(X_val, y_val)
half_ci = 1.96 * np.sqrt(acc_val * (1 - acc_val) / len(y_val))  # intervalle à 95 % sur 300 œuvres


def predict_one(x: torch.Tensor) -> int:
    """Style prédit pour une seule œuvre uint8 (C, H, W), prétraitement compris."""
    return int(logreg.predict(standardize(x[None], MEAN, STD).flatten(1).numpy())[0])


x = X_all[splits["val"][0]]
predict_one(x)  # premier appel hors mesure
latencies = []
for _ in range(200):
    t0 = time.perf_counter()
    predict_one(x)
    latencies.append(time.perf_counter() - t0)

print(f"C retenu : {best_C:g}")
print(f"Accuracy de validation : {acc_val:.3f} ({int(round(acc_val * len(y_val)))}/{len(y_val)}, ± {half_ci:.3f} à 95 %), "
      f"hasard : {1 / len(CLASSES):.2f}")
print(f"Entraînement : {fit_s:.1f} s ; inférence : {1000 * np.median(latencies):.1f} ms par image (médiane sur 200)")
print(f"Mesuré sur CPU : {platform.platform()}")

### 2.3 Sauvegarde

Le modèle est rechargé par les notebooks du ResNet et de la recommandation. Il est sauvegardé dans `modeles/`, qui n'est pas versionné.

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)
logreg_path = MODELS_DIR / "regression_logistique.joblib"
joblib.dump(logreg, logreg_path)

reloaded = joblib.load(logreg_path)
assert (reloaded.predict(X_val) == logreg.predict(X_val)).all()
print(f"{logreg_path.name} : {logreg_path.stat().st_size / 1e6:.1f} Mo, prédictions identiques après rechargement")

### 2.4 Bilan

- **Chiffre à battre : 0,427 d'accuracy de validation** (128 œuvres sur 300, avec C = 1e-5), contre 0,20 au hasard. Sur 300 œuvres, l'intervalle à 95 % fait environ ± 5,6 points : un modèle ne sera dit meilleur que s'il dépasse nettement cet écart.
- **Le réglage de `C` est décisif.** Trop régularisé (C = 1e-7), le modèle sous-apprend : 0,331 en train, 0,313 en validation. Peu régularisé, il apprend le train par cœur : dès C = 1e-3, 0,988 en train mais 0,343 en validation. Avec 49 152 pixels pour 900 œuvres, c'est le sur-apprentissage attendu.
- **Un coût négligeable pour une borne** : quelques secondes d'entraînement, moins d'une milliseconde par image et un modèle de 1,0 Mo. Ces temps sont mesurés sur un ordinateur portable récent ; une borne sera plus lente, mais la marge est large.
- **Sa limite** : il ne combine les pixels que de façon linéaire, chacun à une position fixe. Le réseau dense ajoute des combinaisons non linéaires ; seul le ResNet, convolutif, reconnaîtra un motif où qu'il soit dans l'image.

## 3. Réseau dense de référence

- Réseau feed-forward écrit avec les briques de PyTorch (`nn.Module`, `nn.Linear`, activations) sur les pixels aplatis.
- Boucle d'entraînement propre : forward, perte, `zero_grad`, `backward`, `step` ; `eval()` et `no_grad()` pour évaluer. Le réseau et la boucle vont dans `commun.py`, pour être repris par les notebooks suivants.
- Vérifications : perte initiale proche de ln 5 ≈ 1,61, le réseau sait sur-apprendre un seul batch.
- Mesures : accuracy de validation, matrice de confusion, erreur la plus fréquente, comparaison avec la régression logistique.

In [ ]:
# À compléter